# LeetCode 37 — Sudoku Solver (Hard) bằng Python

Notebook này giảng **từ căn bản** bài toán:

```python
class Solution:
    def solveSudoku(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
```

**Mục tiêu học:**

1. Hiểu luật Sudoku và cách biểu diễn bảng bằng `list` 2 chiều trong Python.
2. Thuộc lòng công thức tìm **box** (ô vuông con 3×3): `box = (r // 3) * 3 + c // 3`.
3. Hiểu **đệ quy (recursion)** và **quay lui (backtracking)** qua ví dụ mini trước khi dùng cho Sudoku.
4. Đi qua 4 cách giải, mỗi cách sửa đúng một điểm yếu của cách trước:
   - Cách 1: backtracking cơ bản, kiểm tra bằng cách quét hàng/cột/box.
   - Cách 2: backtracking + `set` để kiểm tra trong `O(1)`.
   - Cách 3: heuristic **MRV** — luôn điền ô ít lựa chọn nhất trước.
   - Cách 4: **bitmask** + MRV — bản gọn và nhanh để nộp LeetCode.
5. Biết đếm số lần thử để **so sánh bằng số liệu**, không chỉ bằng cảm giác.
6. Tự viết lại lời giải, tránh các lỗi kinh điển của backtracking (quên undo, quên `return True`...).

## 1. Đề bài

Viết chương trình giải một câu đố Sudoku bằng cách **điền vào các ô trống**.

Một lời giải Sudoku phải thỏa mãn **tất cả** các quy tắc sau:

1. Mỗi chữ số `1`–`9` xuất hiện **đúng một lần** trong mỗi **hàng**.
2. Mỗi chữ số `1`–`9` xuất hiện **đúng một lần** trong mỗi **cột**.
3. Mỗi chữ số `1`–`9` xuất hiện **đúng một lần** trong mỗi ô vuông con **3×3** (sub-box) — lưới có 9 ô vuông con như vậy.

Ký tự `'.'` biểu thị **ô trống**.

**Example 1**

```text
Input: board = [["5","3",".",".","7",".",".",".","."],
                ["6",".",".","1","9","5",".",".","."],
                [".","9","8",".",".",".",".","6","."],
                ["8",".",".",".","6",".",".",".","3"],
                ["4",".",".","8",".","3",".",".","1"],
                ["7",".",".",".","2",".",".",".","6"],
                [".","6",".",".",".",".","2","8","."],
                [".",".",".","4","1","9",".",".","5"],
                [".",".",".",".","8",".",".","7","9"]]
Output:        [["5","3","4","6","7","8","9","1","2"],
                ["6","7","2","1","9","5","3","4","8"],
                ["1","9","8","3","4","2","5","6","7"],
                ["8","5","9","7","6","1","4","2","3"],
                ["4","2","6","8","5","3","7","9","1"],
                ["7","1","3","9","2","4","8","5","6"],
                ["9","6","1","5","3","7","2","8","4"],
                ["2","8","7","4","1","9","6","3","5"],
                ["3","4","5","2","8","6","1","7","9"]]
Explanation: The input board is shown above and the only valid solution is shown below.
```

**Constraints** (giữ nguyên theo LeetCode):

- `board.length == 9`
- `board[i].length == 9`
- `board[i][j]` is a digit or `'.'`.
- It is **guaranteed** that the input board has only one solution. (Đề **đảm bảo** bảng đầu vào có đúng một lời giải.)

**Hai điều cần để ý ngay từ đầu:**

- Hàm **không trả về gì** (`-> None`). Ta phải **sửa trực tiếp** (in-place) chính `board` được truyền vào.
- Các ô chứa **chuỗi** `"5"`, không phải số nguyên `5`.

In [1]:
from typing import List
import time
import random

EXAMPLE_BOARD = [
    ["5","3",".",".","7",".",".",".","."],
    ["6",".",".","1","9","5",".",".","."],
    [".","9","8",".",".",".",".","6","."],
    ["8",".",".",".","6",".",".",".","3"],
    ["4",".",".","8",".","3",".",".","1"],
    ["7",".",".",".","2",".",".",".","6"],
    [".","6",".",".",".",".","2","8","."],
    [".",".",".","4","1","9",".",".","5"],
    [".",".",".",".","8",".",".","7","9"],
]

EXAMPLE_SOLUTION = [
    ["5","3","4","6","7","8","9","1","2"],
    ["6","7","2","1","9","5","3","4","8"],
    ["1","9","8","3","4","2","5","6","7"],
    ["8","5","9","7","6","1","4","2","3"],
    ["4","2","6","8","5","3","7","9","1"],
    ["7","1","3","9","2","4","8","5","6"],
    ["9","6","1","5","3","7","2","8","4"],
    ["2","8","7","4","1","9","6","3","5"],
    ["3","4","5","2","8","6","1","7","9"],
]


def print_board(board):
    # In bảng 9x9 có vạch chia box cho dễ nhìn
    for r in range(9):
        if r in (3, 6):
            print("------+-------+------")
        parts = []
        for c in range(9):
            if c in (3, 6):
                parts.append("|")
            parts.append(board[r][c])
        print(" ".join(parts))


print_board(EXAMPLE_BOARD)
empty_count = sum(row.count(".") for row in EXAMPLE_BOARD)
print("\nSố ô trống cần điền:", empty_count)

5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
------+-------+------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
------+-------+------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9

Số ô trống cần điền: 51


## 2. Kiến thức nền 1 — `list` 2 chiều và chỉ số `board[r][c]`

`board` là một **list của các list**: list ngoài chứa 9 hàng, mỗi hàng là một list 9 ô.

- `board[r]` → cả **hàng** thứ `r` (đếm từ 0).
- `board[r][c]` → **ô** ở hàng `r`, cột `c`.
- Muốn lấy cả **cột** `c`, ta phải đi qua từng hàng: `[board[r][c] for r in range(9)]`.

Đây là **list comprehension** (cách viết gọn để tạo list): đọc là "lấy `board[r][c]` với mỗi `r` từ 0 đến 8".

In [2]:
board = EXAMPLE_BOARD

print("Hàng 0      :", board[0])
print("Ô (0, 1)    :", board[0][1])
print("Cột 2       :", [board[r][2] for r in range(9)])
print("Kiểu của ô  :", type(board[0][0]))   # str, không phải int!
print("'5' == 5 ?  :", "5" == 5)              # False -> dễ gây bug nếu so sánh nhầm kiểu

Hàng 0      : ['5', '3', '.', '.', '7', '.', '.', '.', '.']
Ô (0, 1)    : 3
Cột 2       : ['.', '.', '8', '.', '.', '.', '.', '.', '.']
Kiểu của ô  : <class 'str'>
'5' == 5 ?  : False


## 3. Kiến thức nền 2 — Ô `(r, c)` thuộc box nào?

Lưới 9×9 chia thành 9 box 3×3, đánh số từ trái sang phải, trên xuống dưới:

```text
 box 0 | box 1 | box 2
-------+-------+-------
 box 3 | box 4 | box 5
-------+-------+-------
 box 6 | box 7 | box 8
```

**Bước 1 — box nằm ở "dải hàng" nào?** Hàng 0,1,2 → dải 0; hàng 3,4,5 → dải 1; hàng 6,7,8 → dải 2. Đó chính là `r // 3` (chia lấy phần nguyên).

**Bước 2 — box nằm ở "dải cột" nào?** Tương tự: `c // 3`.

**Bước 3 — ghép lại:** mỗi dải hàng có 3 box, nên

```python
box = (r // 3) * 3 + c // 3
```

Ví dụ số: ô `(4, 7)` → `r // 3 = 1`, `c // 3 = 2` → `box = 1*3 + 2 = 5`. Đúng: hàng 4 nằm ở giữa, cột 7 nằm bên phải → box 5.

**Góc trên-trái của box** (để quét 9 ô trong box): `start_r = (r // 3) * 3`, `start_c = (c // 3) * 3`. Với `(4, 7)` → góc là `(3, 6)`.

| `r` | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| `r // 3` | 0 | 0 | 0 | 1 | 1 | 1 | 2 | 2 | 2 |
| `(r // 3) * 3` | 0 | 0 | 0 | 3 | 3 | 3 | 6 | 6 | 6 |

In [3]:
def box_id(r: int, c: int) -> int:
    return (r // 3) * 3 + c // 3

# In số box của cả 81 ô: thấy rõ 9 khối 3x3
for r in range(9):
    print(" ".join(str(box_id(r, c)) for c in range(9)))

r, c = 4, 7
print(f"\nÔ ({r},{c}) thuộc box {box_id(r, c)}, góc trên-trái của box là ({(r // 3) * 3},{(c // 3) * 3})")

# Liệt kê 9 ô của box chứa (4, 7) bằng 2 vòng lặp từ góc trên-trái
start_r, start_c = (r // 3) * 3, (c // 3) * 3
cells_in_box = [(i, j) for i in range(start_r, start_r + 3) for j in range(start_c, start_c + 3)]
print(cells_in_box)
print("Giá trị trong box:", [EXAMPLE_BOARD[i][j] for i, j in cells_in_box])

0 0 0 1 1 1 2 2 2
0 0 0 1 1 1 2 2 2
0 0 0 1 1 1 2 2 2
3 3 3 4 4 4 5 5 5
3 3 3 4 4 4 5 5 5
3 3 3 4 4 4 5 5 5
6 6 6 7 7 7 8 8 8
6 6 6 7 7 7 8 8 8
6 6 6 7 7 7 8 8 8

Ô (4,7) thuộc box 5, góc trên-trái của box là (3,6)
[(3, 6), (3, 7), (3, 8), (4, 6), (4, 7), (4, 8), (5, 6), (5, 7), (5, 8)]
Giá trị trong box: ['.', '.', '3', '.', '.', '1', '.', '.', '6']


## 4. Kiến thức nền 3 — Đệ quy (recursion)

**Đệ quy** là khi một hàm **tự gọi lại chính nó** với bài toán nhỏ hơn. Mỗi hàm đệ quy cần 2 phần:

1. **Base case** (trường hợp dừng): bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: làm một phần việc, rồi giao phần còn lại cho lời gọi con.

Mỗi lần gọi được đặt lên **call stack** (ngăn xếp lời gọi). Lời gọi con chạy xong thì quay về đúng chỗ lời gọi cha đang đứng — đây chính là "cơ chế quay lui" miễn phí mà backtracking tận dụng.

Ví dụ: đếm ngược. In thụt lề theo độ sâu để thấy call stack đi xuống rồi quay lên.

In [4]:
def countdown(n: int, depth: int = 0) -> None:
    indent = "    " * depth
    if n == 0:                      # base case
        print(indent + "n = 0 -> dừng, bắt đầu quay về")
        return
    print(indent + f"vào countdown({n})")
    countdown(n - 1, depth + 1)     # bước đệ quy: giao n-1 cho lời gọi con
    print(indent + f"quay về countdown({n})")

countdown(3)

vào countdown(3)
    vào countdown(2)
        vào countdown(1)
            n = 0 -> dừng, bắt đầu quay về
        quay về countdown(1)
    quay về countdown(2)
quay về countdown(3)


## 5. Kiến thức nền 4 — Quay lui (backtracking)

**Backtracking** = thử từng lựa chọn, đi sâu tiếp; nếu đi vào ngõ cụt thì **hoàn tác (undo)** lựa chọn vừa rồi và thử lựa chọn khác.

Khung chuẩn có 3 động tác, nhớ như một câu thần chú **chọn → đi tiếp → bỏ chọn**:

```python
def backtrack(state):
    if state đã hoàn chỉnh:
        return True                 # tìm thấy lời giải
    for choice in các lựa chọn hợp lệ:
        chọn choice                 # 1. sửa state
        if backtrack(state):        # 2. đi tiếp
            return True             #    lời giải đã tìm thấy -> báo lên trên ngay
        bỏ chọn choice              # 3. undo, trả state về như cũ
    return False                    # mọi lựa chọn đều thất bại -> ngõ cụt
```

**Ví dụ mini:** điền 3 ô `[_, _, _]` bằng các số 1, 2, 3 **không trùng nhau**, và ràng buộc thêm: **ô cuối phải là 1**. Ta in ra từng lần chọn/bỏ để thấy cây quyết định.

In [5]:
def fill_three(slots, depth=0):
    indent = "  " * depth
    if len(slots) == 3:
        ok = slots[-1] == 1
        print(indent + f"đủ 3 ô {slots} -> {'HỢP LỆ' if ok else 'sai (ô cuối khác 1)'}")
        return ok
    for x in [1, 2, 3]:
        if x in slots:              # đã dùng rồi -> bỏ qua (cắt nhánh sớm)
            continue
        slots.append(x)             # 1. chọn
        print(indent + f"chọn {x} -> {slots}")
        if fill_three(slots, depth + 1):   # 2. đi tiếp
            return True
        slots.pop()                 # 3. bỏ chọn (undo)
        print(indent + f"bỏ {x}   -> {slots}")
    return False

result = []
fill_three(result)
print("Kết quả:", result)

chọn 1 -> [1]
  chọn 2 -> [1, 2]
    chọn 3 -> [1, 2, 3]
      đủ 3 ô [1, 2, 3] -> sai (ô cuối khác 1)
    bỏ 3   -> [1, 2]
  bỏ 2   -> [1]
  chọn 3 -> [1, 3]
    chọn 2 -> [1, 3, 2]
      đủ 3 ô [1, 3, 2] -> sai (ô cuối khác 1)
    bỏ 2   -> [1, 3]
  bỏ 3   -> [1]
bỏ 1   -> []
chọn 2 -> [2]
  chọn 1 -> [2, 1]
    chọn 3 -> [2, 1, 3]
      đủ 3 ô [2, 1, 3] -> sai (ô cuối khác 1)
    bỏ 3   -> [2, 1]
  bỏ 1   -> [2]
  chọn 3 -> [2, 3]
    chọn 1 -> [2, 3, 1]
      đủ 3 ô [2, 3, 1] -> HỢP LỆ
Kết quả: [2, 3, 1]


**Quan sát:**

- `[1, 2, 3]` và `[1, 3, 2]` đều sai → hàm **bỏ** số 1 ở ô đầu và thử `2`.
- Nhờ `slots.pop()`, mỗi khi quay về, `slots` **đúng y như trước khi chọn**. Nếu quên undo, `slots` sẽ dài dần và mọi thứ sai bét.
- Khi tìm thấy `[2, 3, 1]`, các lời gọi trả `True` dây chuyền lên trên → **dừng ngay**, không undo lời giải.

Sudoku chính là bài này phóng to: "ô" là các ô trống, "lựa chọn" là chữ số 1–9, "ràng buộc" là luật hàng/cột/box.

### Vì sao không "thử hết mọi cách điền" rồi mới kiểm tra?

Ví dụ của đề có 51 ô trống → số cách điền là `9^51 ≈ 4.6 × 10^48`. Kể cả kiểm tra 1 tỷ cách mỗi giây cũng cần khoảng `10^32` năm.

Backtracking khác ở chỗ: **kiểm tra ngay khi vừa đặt một chữ số**. Hễ một chữ số phạm luật là cắt cả nhánh phía sau nó, giống `if x in slots: continue` trong `fill_three`.

## 6. Cách 1 — Backtracking cơ bản (quét để kiểm tra)

**Ý tưởng tự nhiên** (giống người mới tự giải Sudoku bằng bút chì):

1. Tìm ô trống **đầu tiên** (quét từ trên xuống, trái sang phải).
2. Thử lần lượt `"1"`, `"2"`, ..., `"9"`. Với mỗi chữ số, **kiểm tra hợp lệ**: hàng, cột, box chưa có chữ số đó.
3. Hợp lệ → đặt vào, gọi đệ quy giải phần còn lại.
4. Không còn ô trống → xong (`return True`).
5. Thử hết 9 chữ số mà không được → ngõ cụt (`return False`), ô này trả về `'.'` cho cấp trên thử chữ số khác.

Trước hết, hàm **kiểm tra hợp lệ** `is_valid_placement(board, r, c, d)`: quét 9 ô của hàng, 9 ô của cột, 9 ô của box.

In [6]:
DIGITS = "123456789"

def is_valid_placement(board, r: int, c: int, d: str) -> bool:
    for i in range(9):
        if board[r][i] == d:        # hàng r đã có d
            return False
        if board[i][c] == d:        # cột c đã có d
            return False
    start_r, start_c = (r // 3) * 3, (c // 3) * 3
    for i in range(start_r, start_r + 3):
        for j in range(start_c, start_c + 3):
            if board[i][j] == d:    # box đã có d
                return False
    return True

# Ô (0, 2) của ví dụ: hàng 0 có 5,3,7 ; cột 2 có 8 ; box 0 có 5,3,6,9,8
for d in DIGITS:
    print(d, "->", is_valid_placement(EXAMPLE_BOARD, 0, 2, d))

1 -> True
2 -> True
3 -> False
4 -> True
5 -> False
6 -> False
7 -> False
8 -> False
9 -> False


Kết quả khớp tính tay: ô `(0, 2)` chỉ có thể là `1`, `2` hoặc `4`. Đáp án thật là `4`, nhưng thuật toán **không biết trước** — nó sẽ thử `1` trước, đi sâu, gặp ngõ cụt, quay lui, rồi mới tới `4`.

Giờ ghép thành bộ giải. Vài chi tiết Python:

- `find_empty(board)` trả về `(r, c)` của ô trống đầu tiên, hoặc `None` nếu bảng đã đầy.
- Hàm con `backtrack()` nằm **bên trong** hàm chính để dùng chung `board` và biến đếm `attempts`.
- `nonlocal attempts` cho phép hàm con **sửa** biến `attempts` của hàm cha (không có `nonlocal`, Python sẽ coi `attempts` là biến mới của hàm con).
- `attempts` = **số lần đặt thử** một chữ số. Đây là thước đo công việc ta sẽ dùng để so sánh các cách.
- Tham số `max_attempts` (tùy chọn) để **dừng sớm** khi chạy quá lâu — chỉ dùng cho thí nghiệm.
- Hàm trả về `(solved, attempts)` để ta quan sát; bản LeetCode thì không trả gì.

In [7]:
def find_empty(board):
    for r in range(9):
        for c in range(9):
            if board[r][c] == ".":
                return r, c
    return None


def solve_sudoku_basic(board, max_attempts=None):
    attempts = 0

    def backtrack() -> bool:
        nonlocal attempts
        cell = find_empty(board)
        if cell is None:                    # không còn ô trống -> đã giải xong
            return True
        r, c = cell
        for d in DIGITS:
            if not is_valid_placement(board, r, c, d):
                continue
            if max_attempts is not None and attempts >= max_attempts:
                return False                # hết "ngân sách" thí nghiệm -> bỏ cuộc
            attempts += 1
            board[r][c] = d                 # 1. chọn
            if backtrack():                 # 2. đi tiếp
                return True
            board[r][c] = "."               # 3. bỏ chọn (undo)
        return False                        # 9 chữ số đều thất bại -> ngõ cụt

    solved = backtrack()
    return solved, attempts


board = [row[:] for row in EXAMPLE_BOARD]   # copy để không phá bảng gốc
solved, attempts = solve_sudoku_basic(board)
print("Giải được?", solved, "| số lần đặt thử:", attempts)
print("Khớp Output của đề?", board == EXAMPLE_SOLUTION)
print_board(board)

Giải được? True | số lần đặt thử: 4208
Khớp Output của đề? True
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
------+-------+------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
------+-------+------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9


**Lưu ý `[row[:] for row in EXAMPLE_BOARD]`:** đây là cách **copy** bảng 2 chiều. Mỗi `row[:]` tạo một hàng mới. Nếu chỉ viết `EXAMPLE_BOARD.copy()` thì list ngoài mới nhưng các hàng bên trong **vẫn dùng chung** → sửa bản copy sẽ sửa luôn bản gốc (xem mục Lỗi thường gặp).

Hàm giải xong với **4208 lần đặt thử**, trong khi chỉ có 51 ô trống. Nghĩa là thuật toán đã đặt sai rồi quay lui rất nhiều lần. Để **thấy tận mắt** quay lui, ta trace trên một Sudoku nhỏ hơn.

## 7. Trace từng bước trên Sudoku mini 4×4

Sudoku 4×4: chữ số `1`–`4`, mỗi hàng/cột/box 2×2 có đủ 1–4. Công thức box đổi thành `(r // 2) * 2 + c // 2` (thay 3 bằng 2 — cùng một tư duy).

```text
1 2 | . 4
. . | 2 .
----+----
. . | 1 3
. . | 4 .
```

Câu đố này được chọn vì thuật toán đi sai ở `(1, 0)` từ rất sớm nhưng phải tới hàng 2 mới phát hiện → phải quay lui **3 tầng**.

| Bước | Hành động | Lý do |
|---:|---|---|
| 1 | đặt `3` vào (0,2) | hàng 0 có 1,2,4 → chỉ còn 3 |
| 2 | đặt `3` vào (1,0) | hàng 1 có 2; cột 0 có 1; box 0 có 1,2 → còn {3,4}, thử 3 trước |
| 3 | đặt `4` vào (1,1) | hàng 1 có 3,2; box 0 có 1,2,3 → chỉ còn 4 |
| 4 | đặt `1` vào (1,3) | hàng 1 có 3,4,2 → chỉ còn 1 |
| 5 | đặt `2` vào (2,0) | hàng 2 có 1,3; cột 0 có 1,3 → còn {2,4}, thử 2 |
| 6 | ngõ cụt (2,1) | hàng 2 có 2,1,3 → cần 4, nhưng cột 1 đã có 4 (vừa đặt ở bước 3) |
| 7 | gỡ `2` khỏi (2,0) | quay lui 1 tầng |
| 8 | đặt `4` vào (2,0) | lựa chọn còn lại |
| 9 | ngõ cụt (2,1) | hàng 2 có 4,1,3 → cần 2, nhưng cột 1 đã có 2 |
| 10 | gỡ `4` khỏi (2,0) | (2,0) hết lựa chọn |
| 11 | ngõ cụt (2,0) | báo lên trên |
| 12–16 | gỡ (1,3), ngõ cụt (1,3), gỡ (1,1), ngõ cụt (1,1), gỡ `3` khỏi (1,0) | lùi về tận (1,0) — gốc rễ của sai lầm |
| 17 | đặt `4` vào (1,0) | lựa chọn thứ hai của (1,0) |
| 18–24 | đặt tiếp 7 ô, không còn ngõ cụt | tới khi bảng đầy |

Hàm `trace_backtracking_mini` dưới đây in đúng các bước đó, thụt lề theo độ sâu đệ quy.

In [8]:
def trace_backtracking_mini(rows_text):
    board = [list(row) for row in rows_text]
    n = len(board)                 # 4
    k = int(n ** 0.5)              # cạnh box = 2
    digits = [str(d) for d in range(1, n + 1)]
    step = 0

    def ok(r, c, d):
        if any(board[r][i] == d or board[i][c] == d for i in range(n)):
            return False
        sr, sc = (r // k) * k, (c // k) * k
        return all(board[i][j] != d for i in range(sr, sr + k) for j in range(sc, sc + k))

    def log(depth, text):
        nonlocal step
        step += 1
        print(f"{step:>2}. " + "  " * depth + text)

    def backtrack(depth):
        for r in range(n):
            for c in range(n):
                if board[r][c] == ".":
                    for d in digits:
                        if ok(r, c, d):
                            log(depth, f"đặt {d} vào ({r},{c})")
                            board[r][c] = d
                            if backtrack(depth + 1):
                                return True
                            board[r][c] = "."
                            log(depth, f"gỡ {d} khỏi ({r},{c})")
                    log(depth, f"ngõ cụt tại ({r},{c}) -> quay lui")
                    return False
        return True

    backtrack(0)
    print("Kết quả:", ["".join(row) for row in board])

trace_backtracking_mini(["12.4", "..2.", "..13", "..4."])

 1. đặt 3 vào (0,2)
 2.   đặt 3 vào (1,0)
 3.     đặt 4 vào (1,1)
 4.       đặt 1 vào (1,3)
 5.         đặt 2 vào (2,0)
 6.           ngõ cụt tại (2,1) -> quay lui
 7.         gỡ 2 khỏi (2,0)
 8.         đặt 4 vào (2,0)
 9.           ngõ cụt tại (2,1) -> quay lui
10.         gỡ 4 khỏi (2,0)
11.         ngõ cụt tại (2,0) -> quay lui
12.       gỡ 1 khỏi (1,3)
13.       ngõ cụt tại (1,3) -> quay lui
14.     gỡ 4 khỏi (1,1)
15.     ngõ cụt tại (1,1) -> quay lui
16.   gỡ 3 khỏi (1,0)
17.   đặt 4 vào (1,0)
18.     đặt 3 vào (1,1)
19.       đặt 1 vào (1,3)
20.         đặt 2 vào (2,0)
21.           đặt 4 vào (2,1)
22.             đặt 3 vào (3,0)
23.               đặt 1 vào (3,1)
24.                 đặt 2 vào (3,3)
Kết quả: ['1234', '4321', '2413', '3142']


## 8. Trace trên bảng 9×9 của đề (in gọn)

Bảng thật có hàng nghìn bước, in hết sẽ rất dài. Hàm trace dưới đây chỉ **in 12 sự kiện đầu**, phần còn lại chỉ **đếm**: số lần đặt, số lần gỡ, độ sâu đệ quy lớn nhất.

Để ý: sự kiện đầu tiên là đặt `1` vào (0,2) — nhưng đáp án thật ở đó là `4`. Thuật toán sẽ phải quay lui về (0,2) hai lần.

In [9]:
def trace_basic_9x9(board, show=12):
    board = [row[:] for row in board]
    stats = {"đặt": 0, "gỡ": 0, "ngõ cụt": 0, "sâu nhất": 0}
    shown = 0
    first_cell_history = []        # các chữ số lần lượt đặt vào ô trống đầu tiên

    def log(text):
        nonlocal shown
        if shown < show:
            print(text)
            shown += 1

    first = find_empty(board)

    def backtrack(depth):
        stats["sâu nhất"] = max(stats["sâu nhất"], depth)
        cell = find_empty(board)
        if cell is None:
            return True
        r, c = cell
        for d in DIGITS:
            if is_valid_placement(board, r, c, d):
                stats["đặt"] += 1
                if (r, c) == first:
                    first_cell_history.append(d)
                log(f"độ sâu {depth:>2}: đặt {d} vào ({r},{c})")
                board[r][c] = d
                if backtrack(depth + 1):
                    return True
                board[r][c] = "."
                stats["gỡ"] += 1
                log(f"độ sâu {depth:>2}: gỡ {d} khỏi ({r},{c})")
        stats["ngõ cụt"] += 1
        log(f"độ sâu {depth:>2}: ngõ cụt tại ({r},{c})")
        return False

    backtrack(0)
    print("...")
    print("Thống kê:", stats)
    print(f"Các chữ số lần lượt thử ở ô đầu tiên {first}: {first_cell_history}")
    return board

_ = trace_basic_9x9(EXAMPLE_BOARD)

độ sâu  0: đặt 1 vào (0,2)
độ sâu  1: đặt 2 vào (0,3)
độ sâu  2: đặt 4 vào (0,5)
độ sâu  3: đặt 8 vào (0,6)
độ sâu  4: đặt 9 vào (0,7)
độ sâu  5: ngõ cụt tại (0,8)
độ sâu  4: gỡ 9 khỏi (0,7)
độ sâu  4: ngõ cụt tại (0,7)
độ sâu  3: gỡ 8 khỏi (0,6)
độ sâu  3: đặt 9 vào (0,6)
độ sâu  4: ngõ cụt tại (0,7)
độ sâu  3: gỡ 9 khỏi (0,6)
...
Thống kê: {'đặt': 4208, 'gỡ': 4157, 'ngõ cụt': 4157, 'sâu nhất': 51}
Các chữ số lần lượt thử ở ô đầu tiên (0, 2): ['1', '2', '4']


**Đọc thống kê:**

- `đặt - gỡ = 51`: đúng bằng số ô trống — mọi lần đặt sai đều đã được gỡ, chỉ 51 lần đặt đúng ở lại.
- `sâu nhất = 51`: lời gọi sâu nhất là lúc đã điền hết 51 ô (base case).
- Ô (0,2) đã thử `1` → `2` → `4`: hai nhánh đầu là hai "cây con" khổng lồ bị bỏ đi.

### Độ phức tạp Cách 1

Gọi `m` là số ô trống (`m ≤ 81`).

- **Time:** trường hợp xấu nhất cây tìm kiếm có tới `O(9^m)` nút. Tại **mỗi** nút, ta tốn `O(81)` để `find_empty` quét lại từ đầu, cộng `9 × 27` phép so sánh cho 9 lần `is_valid_placement`. (Vì bảng luôn là 9×9 nên về hình thức có thể nói `O(1)`, nhưng con số thực tế mới là điều đáng quan tâm.)
- **Space:** `O(m)` cho call stack (mỗi tầng đệ quy điền 1 ô); bảng được sửa tại chỗ.

### Cách 1 vướng ở đâu?

1. **Kiểm tra chậm:** mỗi lần hỏi "chữ số `d` đặt được không?" phải quét lại 27 ô — dù hàng/cột/box gần như không đổi giữa hai lần hỏi.
2. **Tìm ô trống chậm:** mỗi lần đệ quy lại quét từ `(0,0)` dù các ô phía trước chắc chắn đã đầy.
3. **Thứ tự điền "mù":** luôn điền theo thứ tự đọc sách, kể cả khi có ô chỉ còn đúng 1 lựa chọn ở chỗ khác.

Cách 2 xử lý (1) và (2). Cách 3 xử lý (3) — cái đắt giá nhất.

## 9. Cách 2 — Backtracking + `set` (kiểm tra O(1))

### Kiến thức nền 5 — `set` (tập hợp)

`set` là cấu trúc lưu các phần tử **không trùng lặp**, hỗ trợ:

| Thao tác | Cú pháp | Thời gian trung bình |
|---|---|---|
| Tạo rỗng | `s = set()` | |
| Thêm | `s.add(x)` | `O(1)` |
| Xóa (phải có sẵn) | `s.remove(x)` | `O(1)` |
| Kiểm tra có không | `x in s` | `O(1)` |

`O(1)` trung bình nghĩa là không cần duyệt từng phần tử như list — Python băm (hash) `x` để nhảy thẳng tới vị trí của nó.

**Ý tưởng:** thay vì mỗi lần quét 27 ô, ta **ghi nhớ sẵn** "hàng `r` đã có những chữ số nào" trong `rows[r]`, tương tự `cols[c]` và `boxes[b]`. Khi đặt hoặc gỡ một chữ số, ta cập nhật 3 set này.

**Cẩn thận:** tạo 9 set rời nhau phải viết `[set() for _ in range(9)]`. Viết `[set()] * 9` sẽ tạo 9 **tham chiếu tới cùng một set**!

In [10]:
s = set()
s.add("5"); s.add("3"); s.add("5")      # thêm "5" lần 2 không có tác dụng
print("s =", s, "| '5' in s:", "5" in s, "| '9' in s:", "9" in s)
s.remove("5")
print("sau remove('5'):", s)

wrong = [set()] * 3
wrong[0].add("x")
print("[set()] * 3 sau khi thêm vào wrong[0]:", wrong)   # cả 3 đều có 'x' -> sai
right = [set() for _ in range(3)]
right[0].add("x")
print("[set() for _ in range(3)]        :", right)

s = {'5', '3'} | '5' in s: True | '9' in s: False
sau remove('5'): {'3'}
[set()] * 3 sau khi thêm vào wrong[0]: [{'x'}, {'x'}, {'x'}]
[set() for _ in range(3)]        : [{'x'}, set(), set()]


### Dựng `rows`, `cols`, `boxes` từ bảng đề

Duyệt 81 ô một lần: ô có chữ số → thêm vào 3 set tương ứng; ô trống → ghi tọa độ vào list `empties`.

Với 3 set, câu hỏi "ô `(r, c)` còn điền được những chữ số nào?" trở thành phép **hiệu tập hợp**: `{1..9} − rows[r] − cols[c] − boxes[b]`.

In [11]:
def build_sets(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empties = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empties.append((r, c))
            else:
                rows[r].add(v)
                cols[c].add(v)
                boxes[box_id(r, c)].add(v)
    return rows, cols, boxes, empties

rows, cols, boxes, empties = build_sets(EXAMPLE_BOARD)
print("rows[0]  =", sorted(rows[0]))
print("cols[2]  =", sorted(cols[2]))
print("boxes[0] =", sorted(boxes[0]))
print("Ứng viên cho (0,2):", sorted(set(DIGITS) - rows[0] - cols[2] - boxes[0]))
print("5 ô trống đầu tiên:", empties[:5], "... tổng", len(empties))

rows[0]  = ['3', '5', '7']
cols[2]  = ['8']
boxes[0] = ['3', '5', '6', '8', '9']
Ứng viên cho (0,2): ['1', '2', '4']
5 ô trống đầu tiên: [(0, 2), (0, 3), (0, 5), (0, 6), (0, 7)] ... tổng 51


### Code Cách 2

Hai thay đổi so với Cách 1:

1. Kiểm tra hợp lệ = 3 phép `in` trên set → `O(1)`.
2. Đệ quy theo **chỉ số** `i` trong `empties`: `backtrack(i)` lo điền ô `empties[i]`, base case là `i == len(empties)`. Không cần quét tìm ô trống nữa.

Quy tắc **đối xứng** khi chọn/bỏ chọn: cập nhật 4 thứ (bảng + 3 set) lúc đặt thì phải hoàn tác **đúng 4 thứ đó** lúc gỡ.

In [12]:
def solve_sudoku_sets(board, max_attempts=None):
    rows, cols, boxes, empties = build_sets(board)
    attempts = 0

    def backtrack(i: int) -> bool:
        nonlocal attempts
        if i == len(empties):               # đã điền hết các ô trống
            return True
        r, c = empties[i]
        b = box_id(r, c)
        for d in DIGITS:
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            if max_attempts is not None and attempts >= max_attempts:
                return False
            attempts += 1
            board[r][c] = d                 # chọn: cập nhật 4 thứ
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack(i + 1):
                return True
            board[r][c] = "."               # bỏ chọn: hoàn tác đúng 4 thứ
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
        return False

    return backtrack(0), attempts


board = [row[:] for row in EXAMPLE_BOARD]
solved, attempts = solve_sudoku_sets(board)
print("Giải được?", solved, "| số lần đặt thử:", attempts, "| khớp đề?", board == EXAMPLE_SOLUTION)

Giải được? True | số lần đặt thử: 4208 | khớp đề? True


**Cùng 4208 lần đặt thử như Cách 1** — không phải trùng hợp: hai cách thử **cùng thứ tự ô** (thứ tự đọc sách) và **cùng thứ tự chữ số**, nên đi qua đúng cùng một cây tìm kiếm. Cách 2 chỉ làm **mỗi bước rẻ hơn**.

Để so sánh, ta thêm 2 bảng đề khác (đều có **đúng một lời giải**, như đề đảm bảo):

- `HARD_BOARD`: khó hơn ví dụ một chút.
- `EVIL_BOARD`: chỉ có 17 ô cho sẵn, được thiết kế để làm khó thứ tự điền "đọc sách".

Hàm `parse_board` đổi chuỗi 81 ký tự thành bảng 9×9 để code gọn.

In [13]:
def parse_board(s: str):
    return [list(s[r * 9:(r + 1) * 9]) for r in range(9)]

HARD_BOARD = parse_board("..9748...7.........2.1.9.....7...24..64.1.59..98...3.....8.3.2.........6...2759..")
EVIL_BOARD = parse_board("4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......")

def run(solver, puzzle, **kwargs):
    board = [row[:] for row in puzzle]
    t0 = time.perf_counter()
    solved, attempts = solver(board, **kwargs)
    ms = (time.perf_counter() - t0) * 1000
    return solved, attempts, ms

for name, puzzle in [("EXAMPLE", EXAMPLE_BOARD), ("HARD", HARD_BOARD)]:
    for solver in (solve_sudoku_basic, solve_sudoku_sets):
        solved, attempts, ms = run(solver, puzzle)
        print(f"{name:<8} {solver.__name__:<20} solved={solved}  attempts={attempts:>6}  time={ms:7.1f} ms")

# Với EVIL_BOARD: cho mỗi cách tối đa 200 000 lần đặt thử
LIMIT = 200_000
for solver in (solve_sudoku_basic, solve_sudoku_sets):
    solved, attempts, ms = run(solver, EVIL_BOARD, max_attempts=LIMIT)
    status = "giải xong" if solved else "CHƯA XONG, bỏ cuộc"
    print(f"EVIL     {solver.__name__:<20} sau {attempts:>7} lần thử ({ms:6.0f} ms): {status}")

EXAMPLE  solve_sudoku_basic   solved=True  attempts=  4208  time=   20.8 ms
EXAMPLE  solve_sudoku_sets    solved=True  attempts=  4208  time=    2.6 ms


HARD     solve_sudoku_basic   solved=True  attempts= 13528  time=   65.4 ms


HARD     solve_sudoku_sets    solved=True  attempts= 13528  time=    9.4 ms


EVIL     solve_sudoku_basic   sau  200000 lần thử (  1059 ms): CHƯA XONG, bỏ cuộc
EVIL     solve_sudoku_sets    sau  200000 lần thử (   139 ms): CHƯA XONG, bỏ cuộc


Trên `EXAMPLE` và `HARD`: số lần thử y hệt nhau, còn thời gian của bản `set` nhỏ hơn rõ rệt (con số ms cụ thể tùy máy).

Nhưng với `EVIL_BOARD`, thứ tự "đọc sách" là thảm họa: hết 200 000 lần thử mà cả hai vẫn chưa xong.

**Bài học:** làm mỗi bước nhanh hơn (Cách 2) là chưa đủ. Với `EVIL_BOARD`, thứ tự cố định cần tới **hàng triệu** lần thử (trên máy thường mất cỡ một phút cho Cách 1). Muốn tiến xa, phải giảm **số bước** — tức là thu nhỏ cây tìm kiếm.

### Độ phức tạp Cách 2

- **Time:** vẫn `O(9^m)` nút trong trường hợp xấu nhất, nhưng mỗi nút chỉ tốn `O(9)` (9 chữ số × kiểm tra `O(1)`), thay vì `O(81 + 9·27)`.
- **Space:** `O(m)` call stack + `empties`, cộng 27 set (mỗi set ≤ 9 phần tử) — hằng số.

## 10. Cách 3 — Heuristic MRV: điền ô ít lựa chọn nhất trước

Người giải Sudoku giỏi không điền theo thứ tự đọc sách. Họ tìm ô **chỉ còn 1 khả năng** và điền nó trước, vì ô đó **chắc chắn đúng**.

Đây là heuristic **MRV — Minimum Remaining Values** (giá trị còn lại ít nhất): ở mỗi bước, chọn ô trống có **ít ứng viên nhất** để rẽ nhánh. (Heuristic = mẹo chọn thứ tự thông minh; nó không đổi tính đúng, chỉ đổi tốc độ.)

Vì sao hiệu quả?

- Ô có **1** ứng viên → không rẽ nhánh, đi thẳng. Cây không phình ra.
- Ô có **0** ứng viên → phát hiện ngõ cụt **ngay lập tức**, thay vì đi sâu thêm hàng chục tầng mới thấy (nguyên tắc **fail-first**: thất bại càng sớm càng rẻ).
- Phải rẽ nhánh thì rẽ ở ô 2 ứng viên (đoán đúng 50%) chứ không phải ô 6 ứng viên.

Thử đếm số ứng viên của từng ô trống trên bảng đề:

In [14]:
def candidate_count_grid(board):
    rows, cols, boxes, _ = build_sets(board)
    for r in range(9):
        line = []
        for c in range(9):
            if board[r][c] != ".":
                line.append("·")      # ô đã có số
            else:
                cand = set(DIGITS) - rows[r] - cols[c] - boxes[box_id(r, c)]
                line.append(str(len(cand)))
        print(" ".join(line))

candidate_count_grid(EXAMPLE_BOARD)

· · 3 2 · 4 4 4 3
· 3 3 · · · 4 3 4
2 · · 2 2 2 5 · 3
· 3 4 3 · 3 4 4 ·
· 2 4 · 1 · 3 3 ·
· 2 4 2 · 2 4 3 ·
3 · 6 3 2 1 · · 1
2 3 3 · · · 2 1 ·
3 4 5 4 · 2 4 · ·


Có những ô chỉ có **1** ứng viên (ví dụ (4,4) chỉ có thể là `5`). Cách 1 lại bắt đầu ở (0,2) với 3 ứng viên — rẽ nhánh sớm và sai.

### Code Cách 3

Ở mỗi lần đệ quy:

1. Duyệt các ô trong `empties` **còn trống**, tính danh sách ứng viên, giữ lại ô có ít ứng viên nhất.
2. Gặp ô có `≤ 1` ứng viên thì dừng tìm luôn (không thể tốt hơn).
3. Không còn ô trống → `return True`.
4. Thử lần lượt các ứng viên của ô được chọn (danh sách rỗng → vòng `for` không chạy → `return False` ngay: đó là fail-first).

In [15]:
def solve_sudoku_mrv(board):
    rows, cols, boxes, empties = build_sets(board)
    attempts = 0

    def candidates(r, c):
        b = box_id(r, c)
        return [d for d in DIGITS if d not in rows[r] and d not in cols[c] and d not in boxes[b]]

    def backtrack() -> bool:
        nonlocal attempts
        best_cell, best_cands = None, None
        for r, c in empties:
            if board[r][c] != ".":
                continue
            cands = candidates(r, c)
            if best_cands is None or len(cands) < len(best_cands):
                best_cell, best_cands = (r, c), cands
                if len(cands) <= 1:          # 0 hoặc 1: không thể tốt hơn
                    break
        if best_cell is None:                # không còn ô trống
            return True
        r, c = best_cell
        b = box_id(r, c)
        for d in best_cands:                 # rỗng -> return False ngay (fail-first)
            attempts += 1
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack():
                return True
            board[r][c] = "."
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
        return False

    return backtrack(), attempts


for name, puzzle in [("EXAMPLE", EXAMPLE_BOARD), ("HARD", HARD_BOARD), ("EVIL", EVIL_BOARD)]:
    solved, attempts, ms = run(solve_sudoku_mrv, puzzle)
    print(f"{name:<8} solved={solved}  attempts={attempts:>5}  time={ms:7.1f} ms")

EXAMPLE  solved=True  attempts=   51  time=    0.3 ms
HARD     solved=True  attempts=  115  time=    0.7 ms
EVIL     solved=True  attempts=  718  time=    7.9 ms


Kết quả rất đáng chú ý:

- `EXAMPLE`: chỉ **51 lần đặt thử** cho 51 ô trống → **không đoán sai lần nào**! Lúc nào cũng có ô chỉ còn 1 ứng viên.
- `HARD`: từ 13 528 xuống **115** lần thử.
- `EVIL`: từ "hơn 200 000 mà chưa xong" xuống chỉ **718** lần thử.

### Trace MRV: thuật toán chọn ô nào?

In 10 quyết định đầu trên `HARD_BOARD`, kèm số ứng viên của ô được chọn. Cuối cùng thống kê: bao nhiêu lần "bị ép" (1 ứng viên) và bao nhiêu lần phải đoán (≥ 2 ứng viên).

In [16]:
def trace_mrv(puzzle, show=10):
    board = [row[:] for row in puzzle]
    rows, cols, boxes, empties = build_sets(board)
    stats = {"bị ép (1 ứng viên)": 0, "phải đoán (>=2)": 0, "ngõ cụt (0)": 0}
    shown = 0

    def backtrack() -> bool:
        nonlocal shown
        best_cell, best_cands = None, None
        for r, c in empties:
            if board[r][c] != ".":
                continue
            b = box_id(r, c)
            cands = [d for d in DIGITS if d not in rows[r] and d not in cols[c] and d not in boxes[b]]
            if best_cands is None or len(cands) < len(best_cands):
                best_cell, best_cands = (r, c), cands
                if len(cands) <= 1:
                    break
        if best_cell is None:
            return True
        k = len(best_cands)
        key = "ngõ cụt (0)" if k == 0 else ("bị ép (1 ứng viên)" if k == 1 else "phải đoán (>=2)")
        stats[key] += 1
        if shown < show:
            print(f"chọn ô {best_cell}: {k} ứng viên {best_cands}")
            shown += 1
        r, c = best_cell
        b = box_id(r, c)
        for d in best_cands:
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack():
                return True
            board[r][c] = "."
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
        return False

    backtrack()
    print("...")
    print("Thống kê các lần chọn ô:", stats)

trace_mrv(HARD_BOARD)

chọn ô (3, 5): 1 ứng viên ['6']
chọn ô (1, 5): 1 ứng viên ['2']
chọn ô (4, 3): 1 ứng viên ['3']
chọn ô (4, 0): 1 ứng viên ['2']
chọn ô (4, 5): 1 ứng viên ['7']
chọn ô (4, 8): 1 ứng viên ['8']
chọn ô (3, 8): 1 ứng viên ['1']
chọn ô (5, 5): 1 ứng viên ['4']
chọn ô (5, 3): 1 ứng viên ['5']
chọn ô (1, 3): 1 ứng viên ['6']
...
Thống kê các lần chọn ô: {'bị ép (1 ứng viên)': 109, 'phải đoán (>=2)': 3, 'ngõ cụt (0)': 3}


**Nhận xét:** phần lớn các bước là "bị ép" — giống người giải Sudoku điền những ô hiển nhiên. Chỉ thỉnh thoảng mới phải đoán, và khi đoán sai thì ngõ cụt (0 ứng viên) lộ ra rất nhanh.

### Độ phức tạp Cách 3

- **Time:** về lý thuyết trường hợp xấu nhất vẫn là hàm mũ theo `m` (Sudoku tổng quát là bài toán khó — NP-complete), nhưng cây tìm kiếm **nhỏ hơn rất nhiều** trên thực tế. Mỗi nút tốn `O(m · 9)` để tìm ô ít ứng viên nhất — đắt hơn Cách 2, nhưng đổi lại số nút giảm hàng trăm, hàng nghìn lần.
- **Space:** `O(m)` call stack + các set — như Cách 2.

### Còn vướng gì?

Phần tốn nhất bây giờ là **tính ứng viên**: mỗi ô tạo một list mới qua 9 chữ số × 3 phép `in`. Cách 4 làm phép tính này bằng **vài phép toán trên số nguyên**.

## 11. Cách 4 — Bitmask + MRV

### Kiến thức nền 6 — Bitmask (mặt nạ bit)

Một số nguyên trong máy tính là dãy **bit** 0/1. Ta dùng **9 bit thấp nhất** của một số để biểu diễn một **tập con** của {1, ..., 9}:

- Bit thứ `d - 1` bật (= 1) ⇔ chữ số `d` có trong tập.
- Ví dụ tập {1, 3, 9}: bật bit 0, 2, 8 → `0b100000101` = 261.

Các phép toán cần biết:

| Phép | Ý nghĩa với tập hợp | Ví dụ |
|---|---|---|
| `1 << k` | số chỉ có bit `k` bật (tập 1 phần tử {k+1}) | `1 << 2` = `0b100` = 4 |
| `a \| b` | hợp (OR) | thêm chữ số vào tập |
| `a & b` | giao (AND) | `mask & (1 << k)` khác 0 ⇔ có chữ số `k+1` |
| `a ^ b` | XOR — bật/tắt bit | tắt bit vừa bật (undo) |
| `~a & 0x1FF` | phần bù trong 9 bit | các chữ số **chưa** dùng (`0x1FF` = 511 = 9 bit 1) |
| `x & -x` | giữ lại **bit 1 thấp nhất** | lấy ứng viên nhỏ nhất |
| `x.bit_length()` | vị trí bit cao nhất + 1 | với `x` chỉ 1 bit: ra đúng chữ số `d` |
| `x.bit_count()` | đếm số bit 1 (Python ≥ 3.10) | số ứng viên |

Vì sao cần `& 0x1FF` sau `~`? Trong Python, `~a` = `-a - 1` là số âm "vô hạn bit 1" ở bên trái; AND với `0x1FF` cắt lại đúng 9 bit.

In [17]:
used = 0
for d in (1, 3, 9):
    used |= 1 << (d - 1)                  # thêm chữ số d vào tập
print("used        =", used, "=", format(used, "09b"))
print("có số 3?    ", bool(used & (1 << 2)), "| có số 2?", bool(used & (1 << 1)))

free = ~used & 0x1FF                      # các chữ số chưa dùng
print("free        =", format(free, "09b"), "-> số ứng viên:", free.bit_count())

low = free & -free                        # bit 1 thấp nhất
print("free & -free=", format(low, "09b"), "-> chữ số", low.bit_length())

# Duyệt mọi chữ số trong free, từ nhỏ tới lớn
digits = []
x = free
while x:
    bit = x & -x
    digits.append(bit.bit_length())
    x ^= bit                              # tắt bit vừa lấy
print("các chữ số trong free:", digits)

used        = 261 = 100000101
có số 3?     True | có số 2? False
free        = 011111010 -> số ứng viên: 6
free & -free= 000000010 -> chữ số 2
các chữ số trong free: [2, 4, 5, 6, 7, 8]


### Áp dụng cho Sudoku

- `rows[r]`, `cols[c]`, `boxes[b]` giờ là **số nguyên** (mask), không còn là set.
- Chữ số đã dùng quanh ô `(r, c)`: `rows[r] | cols[c] | boxes[b]` — **1 phép OR** thay cho 27 lần so sánh.
- Ứng viên: `free = ~(rows[r] | cols[c] | boxes[b]) & 0x1FF`; số ứng viên: `free.bit_count()`.
- Đặt chữ số `d`: `rows[r] |= bit` (với `bit = 1 << (d-1)`). Gỡ: `rows[r] ^= bit`.

Cell đầu tiên dưới đây kiểm tra lại ô (0,2) — phải ra {1, 2, 4} như ở Cách 1. Cell thứ hai là bộ giải Cách 4: cùng logic MRV như Cách 3, chỉ thay set bằng mask. Vì duyệt ứng viên từ bit thấp lên cao (chữ số nhỏ trước) và chọn ô theo cùng quy tắc, số lần thử phải **bằng đúng** Cách 3 — một phép kiểm tra chéo hay.

In [18]:
FULL = 0x1FF

def build_masks(board):
    rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
    empties = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empties.append((r, c))
            else:
                bit = 1 << (int(v) - 1)
                rows[r] |= bit
                cols[c] |= bit
                boxes[box_id(r, c)] |= bit
    return rows, cols, boxes, empties

rows, cols, boxes, _ = build_masks(EXAMPLE_BOARD)
used = rows[0] | cols[2] | boxes[0]
free = ~used & FULL
print("rows[0]  =", format(rows[0], "09b"), " (bit bên phải nhất = chữ số 1)")
print("used     =", format(used, "09b"))
print("free     =", format(free, "09b"), "->", [d for d in range(1, 10) if free & (1 << (d - 1))])

rows[0]  = 001010100  (bit bên phải nhất = chữ số 1)
used     = 111110100
free     = 000001011 -> [1, 2, 4]


In [19]:
def solve_sudoku_bitmask(board):
    rows, cols, boxes, empties = build_masks(board)
    attempts = 0

    def backtrack() -> bool:
        nonlocal attempts
        best_cell, best_free, best_count = None, 0, 10
        for r, c in empties:
            if board[r][c] != ".":
                continue
            free = ~(rows[r] | cols[c] | boxes[box_id(r, c)]) & FULL
            count = free.bit_count()
            if count < best_count:
                best_cell, best_free, best_count = (r, c), free, count
                if count <= 1:
                    break
        if best_cell is None:
            return True
        r, c = best_cell
        b = box_id(r, c)
        free = best_free
        while free:                          # free == 0 -> return False ngay
            bit = free & -free               # ứng viên nhỏ nhất
            free ^= bit
            attempts += 1
            board[r][c] = str(bit.bit_length())
            rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
            if backtrack():
                return True
            board[r][c] = "."
            rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
        return False

    return backtrack(), attempts


for name, puzzle in [("EXAMPLE", EXAMPLE_BOARD), ("HARD", HARD_BOARD), ("EVIL", EVIL_BOARD)]:
    _, a3, _ = run(solve_sudoku_mrv, puzzle)
    solved, a4, ms = run(solve_sudoku_bitmask, puzzle)
    print(f"{name:<8} solved={solved}  attempts={a4:>5}  (Cách 3: {a3:>5})  time={ms:6.1f} ms")

EXAMPLE  solved=True  attempts=   51  (Cách 3:    51)  time=   0.2 ms
HARD     solved=True  attempts=  115  (Cách 3:   115)  time=   0.3 ms
EVIL     solved=True  attempts=  718  (Cách 3:   718)  time=   2.8 ms


### Độ phức tạp Cách 4

- **Time:** cùng cây tìm kiếm với Cách 3; mỗi nút tốn `O(m)` phép toán số nguyên rất rẻ (không tạo list ứng viên).
- **Space:** `O(m)` call stack; 27 số nguyên cho mask.

## 12. Bảng số liệu tổng hợp

Chạy cả 4 cách trên 3 bảng. Cách 1 và 2 bị giới hạn 200 000 lần thử trên `EVIL`.

In [20]:
solvers = [
    ("Cách 1 basic",   solve_sudoku_basic,   True),
    ("Cách 2 sets",    solve_sudoku_sets,    True),
    ("Cách 3 MRV",     solve_sudoku_mrv,     False),
    ("Cách 4 bitmask", solve_sudoku_bitmask, False),
]
puzzles = [("EXAMPLE", EXAMPLE_BOARD), ("HARD", HARD_BOARD), ("EVIL", EVIL_BOARD)]

print(f"{'':<16}" + "".join(f"{name:>22}" for name, _ in puzzles))
for label, solver, has_limit in solvers:
    line = f"{label:<16}"
    for name, puzzle in puzzles:
        kwargs = {"max_attempts": 200_000} if has_limit else {}
        solved, attempts, ms = run(solver, puzzle, **kwargs)
        cell = f"{attempts} lần/{ms:.0f}ms" if solved else f">{attempts} (bỏ cuộc)"
        line += f"{cell:>22}"
    print(line)

                               EXAMPLE                  HARD                  EVIL


Cách 1 basic             4208 lần/20ms        13528 lần/63ms     >200000 (bỏ cuộc)
Cách 2 sets               4208 lần/4ms        13528 lần/10ms     >200000 (bỏ cuộc)
Cách 3 MRV                  51 lần/0ms           115 lần/1ms           718 lần/7ms
Cách 4 bitmask              51 lần/0ms           115 lần/0ms           718 lần/2ms


**Đọc bảng:**

- Cách 1 → 2: **cùng số lần thử**, thời gian giảm (mỗi bước rẻ hơn).
- Cách 2 → 3: **số lần thử giảm mạnh** (cây nhỏ hơn) — đây là cú nhảy lớn nhất.
- Cách 3 → 4: cùng số lần thử, thời gian giảm tiếp (tính ứng viên bằng bit).

Hai trục cải tiến của mọi thuật toán tìm kiếm: **làm ít bước hơn** và **làm mỗi bước rẻ hơn**.

## 13. Code nộp LeetCode

Chọn Cách 4 (bitmask + MRV): ngắn, nhanh, dễ giải thích. So với hàm demo:

- Bỏ biến đếm `attempts`, không `return` gì (signature là `-> None`).
- Sửa **trực tiếp** `board` — LeetCode kiểm tra chính object `board` sau khi hàm chạy xong.
- Đề đảm bảo có đúng một lời giải nên `backtrack()` chắc chắn trả `True`.

Sau cell `Solution` là cell **test**.

Ta viết `is_valid_solution(board, original)` kiểm tra **độc lập** với thuật toán:

1. Mỗi hàng, cột, box là đúng tập `{"1", ..., "9"}`.
2. Mọi ô cho sẵn trong đề được giữ nguyên.

Các ca test: ví dụ của đề (so khớp Output), `HARD`, `EVIL`, bảng đã đầy sẵn (không có ô trống), bảng chỉ thiếu 1 ô, bảng trống cả hàng cuối, và kiểm tra hàm **trả về `None`** và sửa **đúng object** được truyền vào.

In [21]:
class Solution:
    def solveSudoku(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        FULL = 0x1FF
        rows, cols, boxes = [0] * 9, [0] * 9, [0] * 9
        empties = []
        for r in range(9):
            for c in range(9):
                if board[r][c] == ".":
                    empties.append((r, c))
                else:
                    bit = 1 << (int(board[r][c]) - 1)
                    rows[r] |= bit
                    cols[c] |= bit
                    boxes[(r // 3) * 3 + c // 3] |= bit

        def backtrack() -> bool:
            best, best_free, best_count = None, 0, 10
            for r, c in empties:
                if board[r][c] != ".":
                    continue
                free = ~(rows[r] | cols[c] | boxes[(r // 3) * 3 + c // 3]) & FULL
                count = free.bit_count()
                if count < best_count:
                    best, best_free, best_count = (r, c), free, count
                    if count <= 1:
                        break
            if best is None:
                return True
            r, c = best
            b = (r // 3) * 3 + c // 3
            free = best_free
            while free:
                bit = free & -free
                free ^= bit
                board[r][c] = str(bit.bit_length())
                rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
                if backtrack():
                    return True
                board[r][c] = "."
                rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
            return False

        backtrack()

In [22]:
def is_valid_solution(board, original) -> bool:
    target = set(DIGITS)
    for i in range(9):
        if set(board[i]) != target:                          # hàng i
            return False
        if {board[r][i] for r in range(9)} != target:        # cột i
            return False
    for b in range(9):
        sr, sc = (b // 3) * 3, (b % 3) * 3
        if {board[r][c] for r in range(sr, sr + 3) for c in range(sc, sc + 3)} != target:
            return False
    return all(original[r][c] in (".", board[r][c]) for r in range(9) for c in range(9))


def check_solution(name, puzzle, expected=None):
    board = [row[:] for row in puzzle]
    same_object = board
    result = Solution().solveSudoku(board)
    assert result is None, (name, "phải trả về None")
    assert board is same_object, (name, "phải sửa in-place")
    assert is_valid_solution(board, puzzle), (name, "lời giải không hợp lệ")
    if expected is not None:
        assert board == expected, (name, "không khớp expected")
    print(f"{name:<26} PASS")


check_solution("Example 1", EXAMPLE_BOARD, EXAMPLE_SOLUTION)
check_solution("HARD_BOARD", HARD_BOARD)
check_solution("EVIL_BOARD", EVIL_BOARD)
check_solution("bảng đã đầy sẵn", EXAMPLE_SOLUTION, EXAMPLE_SOLUTION)

one_hole = [row[:] for row in EXAMPLE_SOLUTION]
one_hole[8][8] = "."
check_solution("chỉ thiếu 1 ô (8,8)", one_hole, EXAMPLE_SOLUTION)

last_row_empty = [row[:] for row in EXAMPLE_SOLUTION]
last_row_empty[8] = ["."] * 9
check_solution("trống cả hàng cuối", last_row_empty, EXAMPLE_SOLUTION)

Example 1                  PASS
HARD_BOARD                 PASS
EVIL_BOARD                 PASS
bảng đã đầy sẵn            PASS
chỉ thiếu 1 ô (8,8)        PASS
trống cả hàng cuối         PASS


### Đối chiếu chéo trên bảng ngẫu nhiên

Tạo bảng ngẫu nhiên có **đúng một lời giải** (như đề đảm bảo):

1. **Xáo một lời giải có sẵn** mà vẫn hợp lệ: đổi tên chữ số (hoán vị 1–9), hoán vị các hàng **trong cùng dải 3 hàng**, hoán vị các dải, làm tương tự với cột. Các phép này không phá luật Sudoku.
2. **Xóa dần từng ô** theo thứ tự ngẫu nhiên; chỉ giữ việc xóa nếu bảng **vẫn còn đúng 1 lời giải** (dùng hàm đếm nghiệm dừng sớm khi thấy nghiệm thứ 2). Dừng khi đã có 45 ô trống để Cách 1 chạy nhanh.
3. Chạy cả 4 cách + `Solution`: tất cả phải ra **đúng** lời giải gốc. Thêm 2 kiểm tra cấu trúc: Cách 1 và 2 cùng số lần thử; Cách 3 và 4 cùng số lần thử.

`random.seed(37)` giúp kết quả lặp lại được mỗi lần chạy.

In [23]:
def count_solutions_for_test(puzzle, limit=2) -> int:
    board = [row[:] for row in puzzle]
    rows, cols, boxes, empties = build_masks(board)
    found = 0

    def backtrack():
        nonlocal found
        best, best_free, best_count = None, 0, 10
        for r, c in empties:
            if board[r][c] == ".":
                free = ~(rows[r] | cols[c] | boxes[box_id(r, c)]) & FULL
                if free.bit_count() < best_count:
                    best, best_free, best_count = (r, c), free, free.bit_count()
        if best is None:
            found += 1
            return
        r, c = best
        b = box_id(r, c)
        while best_free and found < limit:
            bit = best_free & -best_free
            best_free ^= bit
            board[r][c] = "x"
            rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
            backtrack()
            board[r][c] = "."
            rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit

    backtrack()
    return found


def random_solved_grid(rng):
    relabel = list(DIGITS)
    rng.shuffle(relabel)
    mapping = dict(zip(DIGITS, relabel))

    def shuffled_order():
        bands = [0, 1, 2]
        rng.shuffle(bands)
        order = []
        for band in bands:
            inner = [0, 1, 2]
            rng.shuffle(inner)
            order += [band * 3 + i for i in inner]
        return order

    row_order, col_order = shuffled_order(), shuffled_order()
    return [[mapping[EXAMPLE_SOLUTION[r][c]] for c in col_order] for r in row_order]


def random_unique_puzzle(rng, max_empty=45):
    solution = random_solved_grid(rng)
    puzzle = [row[:] for row in solution]
    cells = [(r, c) for r in range(9) for c in range(9)]
    rng.shuffle(cells)
    removed = 0
    for r, c in cells:
        if removed == max_empty:
            break
        keep = puzzle[r][c]
        puzzle[r][c] = "."
        if count_solutions_for_test(puzzle) == 1:
            removed += 1
        else:
            puzzle[r][c] = keep          # xóa ô này làm mất tính duy nhất -> trả lại
    return puzzle, solution


rng = random.Random(37)
t0 = time.perf_counter()
for t in range(8):
    puzzle, solution = random_unique_puzzle(rng)
    attempts = {}
    for label, solver, _ in solvers:
        board = [row[:] for row in puzzle]
        solved, attempts[label] = solver(board)
        assert solved and board == solution, (t, label)
    board = [row[:] for row in puzzle]
    Solution().solveSudoku(board)
    assert board == solution, (t, "Solution")
    assert attempts["Cách 1 basic"] == attempts["Cách 2 sets"]
    assert attempts["Cách 3 MRV"] == attempts["Cách 4 bitmask"]
    empties_n = sum(row.count(".") for row in puzzle)
    print(f"bảng {t}: {empties_n} ô trống | lần thử Cách 1/2 = {attempts['Cách 1 basic']:>5}, Cách 3/4 = {attempts['Cách 3 MRV']:>3}")
print(f"Tất cả đối chiếu chéo PASS ({time.perf_counter() - t0:.1f} s)")

bảng 0: 45 ô trống | lần thử Cách 1/2 =   787, Cách 3/4 =  48
bảng 1: 45 ô trống | lần thử Cách 1/2 =   266, Cách 3/4 =  45
bảng 2: 45 ô trống | lần thử Cách 1/2 =   133, Cách 3/4 =  45


bảng 3: 45 ô trống | lần thử Cách 1/2 =    56, Cách 3/4 =  45
bảng 4: 45 ô trống | lần thử Cách 1/2 =   192, Cách 3/4 =  45
bảng 5: 45 ô trống | lần thử Cách 1/2 =   114, Cách 3/4 =  45
bảng 6: 45 ô trống | lần thử Cách 1/2 =   126, Cách 3/4 =  45
bảng 7: 45 ô trống | lần thử Cách 1/2 =   160, Cách 3/4 =  55
Tất cả đối chiếu chéo PASS (0.1 s)


## 14. Vì sao thuật toán đúng?

**Invariant (bất biến) 1 — bảng luôn "sạch":** tại mọi thời điểm, các chữ số trên bảng không vi phạm luật nào. Lý do: ta chỉ đặt chữ số đã qua kiểm tra hàng/cột/box. Vì vậy khi **không còn ô trống**, bảng chắc chắn là một lời giải hợp lệ → `return True` an toàn.

**Invariant 2 — undo trả về đúng trạng thái cũ:** sau `chọn → đi tiếp → bỏ chọn`, bảng (và các set/mask) giống hệt trước khi chọn. Vì vậy mỗi lựa chọn kế tiếp được thử trên đúng trạng thái nó cần.

**Không bỏ sót lời giải:** gọi ô đang xét là `X`. Lời giải thật (nếu tồn tại từ trạng thái hiện tại) phải đặt **một** chữ số hợp lệ nào đó vào `X`. Ta thử **mọi** chữ số hợp lệ cho `X`, nên nhánh chứa lời giải thật luôn được ghé. Nếu `X` không có chữ số hợp lệ nào thì trạng thái hiện tại **không thể** mở rộng thành lời giải → quay lui là đúng.

**Vì sao đổi thứ tự ô (MRV) vẫn đúng?** Lập luận trên đúng với **bất kỳ** ô trống `X` nào — sớm muộn gì mọi ô cũng phải được điền. MRV chỉ chọn `X` khéo hơn để cây nhỏ đi, không làm mất nhánh nào.

**Dừng chắc chắn:** mỗi tầng đệ quy điền thêm một ô, số ô hữu hạn, mỗi ô ≤ 9 lựa chọn → cây hữu hạn.

## 15. So sánh các cách

| Cách | Chọn ô | Kiểm tra hợp lệ | Chi phí mỗi nút | Cây tìm kiếm | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Basic | ô trống đầu tiên (quét lại `O(81)`) | quét 27 ô | đắt | lớn | học khái niệm backtracking |
| 2. Sets | ô kế tiếp trong `empties` | 3 phép `in` `O(1)` | rẻ | lớn (= Cách 1) | bản "chuẩn phỏng vấn" dễ viết |
| 3. MRV | ô ít ứng viên nhất | set | trung bình `O(9m)` | **rất nhỏ** | khi đề khó, cần cắt nhánh mạnh |
| 4. Bitmask + MRV | ô ít ứng viên nhất | 1 phép OR | rẻ `O(m)` | rất nhỏ (= Cách 3) | bản nộp nhanh nhất |

Với `m` = số ô trống: Time worst-case của cả 4 cách đều là hàm mũ theo `m` (giới hạn trên `9^m` nút); Space đều `O(m)` cho call stack.

## 16. Lỗi thường gặp

### Lỗi 1 — Sai công thức box

```python
box = r // 3 + c // 3        # SAI
box = (r // 3) * 3 + c // 3  # ĐÚNG
```

Bản sai cho ô (0,3) và (3,0) cùng box `1`, và chỉ sinh ra 5 giá trị 0..4 thay vì 9 box.

In [24]:
wrong_ids = {r // 3 + c // 3 for r in range(9) for c in range(9)}
right_ids = {(r // 3) * 3 + c // 3 for r in range(9) for c in range(9)}
print("Công thức sai sinh ra :", sorted(wrong_ids))
print("Công thức đúng sinh ra:", sorted(right_ids))
print("(0,3) và (3,0) theo công thức sai:", 0 // 3 + 3 // 3, 3 // 3 + 0 // 3, "-> trùng box!")

Công thức sai sinh ra : [0, 1, 2, 3, 4]
Công thức đúng sinh ra: [0, 1, 2, 3, 4, 5, 6, 7, 8]
(0,3) và (3,0) theo công thức sai: 1 1 -> trùng box!


### Lỗi 2 — Gọi đệ quy mà không kiểm tra kết quả

```python
board[r][c] = d
backtrack()            # SAI: bỏ qua kết quả trả về
board[r][c] = "."      # ... nên kể cả khi đã giải xong vẫn chạy undo!
```

Lời giải tìm được bị **gỡ sạch** khi các lời gọi quay về. Phải viết `if backtrack(): return True` để "đóng băng" lời giải.

### Lỗi 3 — Quên undo

```python
board[r][c] = d
if backtrack():
    return True
# SAI: thiếu board[r][c] = "." (và thiếu gỡ khỏi set)
```

Chữ số sai ở lại trên bảng, các nhánh sau làm việc trên trạng thái bẩn. Cùng chạy 2 bản lỗi trên Sudoku mini 4×4 (câu đố cần quay lui) để thấy hậu quả:

In [25]:
def mini_ok(board, r, c, d):
    if any(board[r][i] == d or board[i][c] == d for i in range(4)):
        return False
    sr, sc = (r // 2) * 2, (c // 2) * 2
    return all(board[i][j] != d for i in range(sr, sr + 2) for j in range(sc, sc + 2))

def mini_solver(board, bug):
    for r in range(4):
        for c in range(4):
            if board[r][c] == ".":
                for d in "1234":
                    if mini_ok(board, r, c, d):
                        board[r][c] = d
                        if bug == "không kiểm tra kết quả":
                            mini_solver(board, bug)
                            board[r][c] = "."
                        else:
                            if mini_solver(board, bug):
                                return True
                            if bug != "quên undo":
                                board[r][c] = "."
                return False
    return True

MINI = ["12.4", "..2.", "..13", "..4."]
for bug in ["không lỗi", "không kiểm tra kết quả", "quên undo"]:
    board = [list(row) for row in MINI]
    ret = mini_solver(board, bug)
    print(f"{bug:<24} -> trả về {ret!s:<5} bảng = {[''.join(row) for row in board]}")

không lỗi                -> trả về True  bảng = ['1234', '4321', '2413', '3142']
không kiểm tra kết quả   -> trả về False bảng = ['12.4', '..2.', '..13', '..4.']
quên undo                -> trả về False bảng = ['1234', '3421', '4.13', '..4.']


- **Không lỗi:** ra lời giải `['1234', '4321', '2413', '3142']`.
- **Không kiểm tra kết quả:** bảng quay về **y nguyên đề bài** — lời giải đã bị undo mất.
- **Quên undo:** các chữ số sai (`3`, `4`, `1` ở hàng 1, `4` ở (2,0)) **kẹt lại**; thuật toán coi chúng là ô đã điền nên không bao giờ sửa → kết luận sai "không có lời giải" (`False`) và để lại bảng dở dang.

### Lỗi 4 — Gán lại `board` thay vì sửa tại chỗ

```python
def solveSudoku(self, board):
    solved = ...            # tạo bảng mới đã giải
    board = solved          # SAI: chỉ đổi biến cục bộ, LeetCode vẫn thấy bảng cũ
```

Tên `board` trong hàm chỉ là một "nhãn" trỏ tới list của người gọi. Gán `board = ...` là dán nhãn sang list khác; list gốc không đổi. Phải sửa **nội dung**: `board[r][c] = ...` hoặc `board[:] = solved`.

### Lỗi 5 — Copy nông (shallow copy) bảng 2 chiều

`board.copy()` hay `list(board)` chỉ copy list **ngoài**; 9 hàng bên trong vẫn dùng chung. Khi test, giải trên "bản copy" sẽ phá luôn bảng gốc, các test sau chạy trên bảng đã giải sẵn và PASS "giả". Cell dưới minh họa cả Lỗi 4 và Lỗi 5.

In [26]:
def wrong_inplace(board):
    board = [row[:] for row in EXAMPLE_SOLUTION]      # gán lại biến cục bộ

def right_inplace(board):
    board[:] = [row[:] for row in EXAMPLE_SOLUTION]   # thay nội dung list gốc

b1 = [row[:] for row in EXAMPLE_BOARD]; wrong_inplace(b1)
b2 = [row[:] for row in EXAMPLE_BOARD]; right_inplace(b2)
print("gán lại board = ... -> bảng của người gọi đã giải?", b1 == EXAMPLE_SOLUTION)
print("board[:] = ...      -> bảng của người gọi đã giải?", b2 == EXAMPLE_SOLUTION)

# Lỗi 5: copy nông
shallow = EXAMPLE_BOARD.copy()
deep = [row[:] for row in EXAMPLE_BOARD]
print("hàng 0 dùng chung với bản gốc? shallow:", shallow[0] is EXAMPLE_BOARD[0], "| deep:", deep[0] is EXAMPLE_BOARD[0])

gán lại board = ... -> bảng của người gọi đã giải? False
board[:] = ...      -> bảng của người gọi đã giải? True
hàng 0 dùng chung với bản gốc? shallow: True | deep: False


### Lỗi 6 — Nhầm kiểu `str` và `int`

```python
if board[r][i] == 5:    # SAI: ô chứa "5" (str) -> luôn False -> mọi chữ số đều "hợp lệ"
if board[r][i] == "5":  # ĐÚNG
```

Tương tự, khi ghi kết quả phải ghi **chuỗi**: `board[r][c] = str(d)`, không phải `board[r][c] = d` với `d` là số nguyên.

### Lỗi 7 — Undo không đối xứng

Đặt thì cập nhật bảng + `rows` + `cols` + `boxes`; gỡ phải hoàn tác **đủ cả 4**. Quên một set → set "nhớ nhầm" chữ số đã gỡ → các nhánh sau bị chặn oan, có thể kết luận sai "không có lời giải".

## 17. Mẹo nhận dạng pattern

Nghĩ tới **backtracking** khi đề có dấu hiệu:

- "Điền / sắp xếp / chọn" các phần tử sao cho **thỏa ràng buộc** (constraint satisfaction).
- Cần **một** cấu hình hợp lệ, hoặc **liệt kê / đếm** mọi cấu hình.
- Không gian thử rất lớn, nhưng ràng buộc cho phép **cắt nhánh sớm** khi vừa vi phạm.
- Input nhỏ, cố định (bảng 9×9, n ≤ 9 quân hậu...) — dấu hiệu thuật toán hàm mũ vẫn chấp nhận được.

Và khi đã chọn backtracking, tự hỏi thêm 3 câu:

1. **Kiểm tra nhanh được không?** → lưu trạng thái phụ (set / mảng đánh dấu / bitmask).
2. **Thứ tự thử có khéo hơn được không?** → MRV / fail-first: ô ít lựa chọn nhất trước.
3. **Undo có đối xứng không?** → mọi thứ thay đổi khi "chọn" phải được hoàn tác khi "bỏ chọn".

## 18. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 36 — Valid Sudoku | chỉ **kiểm tra** bảng dở dang có hợp lệ không: dùng lại `rows/cols/boxes` (Bài tập 5) |
| LeetCode 51 / 52 — N-Queens / N-Queens II | backtracking đặt hậu, đánh dấu cột + 2 đường chéo bằng set hoặc bitmask |
| LeetCode 46 — Permutations | khung chọn → đi tiếp → bỏ chọn thuần túy (giống ví dụ `fill_three`) |
| LeetCode 79 — Word Search | backtracking trên lưới, đánh dấu ô đã đi rồi undo |
| Đếm số lời giải / kiểm tra tính duy nhất | không dừng ở lời giải đầu, đếm tới 2 thì dừng (đã dùng trong test ngẫu nhiên) |
| Sudoku N×N tổng quát (4×4, 16×16) | thay 3 bằng `k = √N`, mask `N` bit (Bài tập 6) |
| Đọc thêm | Algorithm X / Dancing Links (Knuth) — mô hình Sudoku thành bài "exact cover" |

## 19. Bài tập

### Bài 1 — Tính tay (đoán output)

Không chạy code, hãy cho biết với mỗi ô, `box_id(r, c)` bằng bao nhiêu và góc trên-trái của box là ô nào:

`(0, 8)`, `(4, 4)`, `(7, 2)`, `(5, 6)`, `(8, 8)`.

### Bài 2 — Tính tay ứng viên

Trên `EXAMPLE_BOARD`, liệt kê chữ số trong **hàng 4**, **cột 4**, **box 4**, rồi suy ra các ứng viên của ô `(4, 4)`. Sau đó làm tương tự cho ô `(6, 0)`.

(Gợi ý kiểm tra: lưới đếm ứng viên ở mục 10 cho biết mỗi ô có bao nhiêu ứng viên.)

### Bài 3 — Điền TODO: hàm kiểm tra hợp lệ

Viết lại `is_valid_placement` bằng cách quét hàng, cột, box. **Không** nhìn lại code ở mục 6.

In [27]:
def exercise_is_valid_placement(board, r: int, c: int, d: str) -> bool:
    # TODO 1: duyệt i từ 0 đến 8, nếu board[r][i] == d hoặc board[i][c] == d -> False
    # TODO 2: tính góc box: start_r = (r // 3) * 3, start_c = (c // 3) * 3
    # TODO 3: duyệt 3x3 ô trong box, gặp d -> False
    # TODO 4: return True
    pass

# Tự làm xong rồi mới mở test:
# assert [d for d in DIGITS if exercise_is_valid_placement(EXAMPLE_BOARD, 0, 2, d)] == ["1", "2", "4"]
# assert exercise_is_valid_placement(EXAMPLE_BOARD, 4, 4, "5") == True
# assert exercise_is_valid_placement(EXAMPLE_BOARD, 4, 4, "6") == False   # cột 4 có 6

### Bài 4 — Điền TODO: backtracking với `set`

Hoàn thiện bộ giải theo Cách 2. Phần dựng set và `empties` đã có; bạn viết `backtrack(i)` với đủ 3 động tác **chọn → đi tiếp → bỏ chọn** và undo đối xứng.

In [28]:
def exercise_solve_sets(board) -> None:
    rows, cols, boxes, empties = build_sets(board)

    def backtrack(i: int) -> bool:
        # TODO 1: base case: i == len(empties) -> True
        # TODO 2: lấy r, c = empties[i] và b = box_id(r, c)
        # TODO 3: với mỗi d trong DIGITS chưa có trong rows[r], cols[c], boxes[b]:
        #           chọn: ghi board + add vào 3 set
        #           đi tiếp: if backtrack(i + 1): return True
        #           bỏ chọn: trả board về "." + remove khỏi 3 set
        # TODO 4: return False
        pass

    backtrack(0)

# Tự làm xong rồi mới mở test:
# b = [row[:] for row in EXAMPLE_BOARD]
# exercise_solve_sets(b)
# assert b == EXAMPLE_SOLUTION

### Bài 5 — Biến thể: LeetCode 36 Valid Sudoku

Cho một bảng **dở dang** (có `'.'`). Trả `True` nếu các chữ số đã có **không vi phạm** luật (không trùng trong hàng/cột/box). Không cần kiểm tra bảng có giải được hay không.

In [29]:
def exercise_is_valid_sudoku(board) -> bool:
    # TODO: tạo rows, cols, boxes là 9 set rỗng mỗi loại
    # TODO: duyệt 81 ô, bỏ qua '.', nếu chữ số đã có trong set tương ứng -> False, ngược lại thêm vào
    # TODO: return True
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_is_valid_sudoku(EXAMPLE_BOARD) == True
# bad = [row[:] for row in EXAMPLE_BOARD]; bad[0][2] = "6"     # box 0 đã có 6
# assert exercise_is_valid_sudoku(bad) == False

### Bài 6 — Mini challenge: Sudoku N×N tổng quát

Viết `exercise_solve_nxn(board)` giải Sudoku cạnh `N = k*k` (ví dụ `N = 4` với box 2×2, `N = 9` với box 3×3). Chữ số là `"1"`..`str(N)` (giới hạn `N ≤ 9` để mỗi ô là một ký tự). Dùng bitmask + MRV.

Yêu cầu: dùng được cho cả Sudoku mini `["12.4", "..2.", "..13", "..4."]` (dưới dạng list các list) và `EXAMPLE_BOARD`.

In [30]:
def exercise_solve_nxn(board) -> None:
    # TODO 1: n = len(board), k = int(n ** 0.5), FULL_N = (1 << n) - 1
    # TODO 2: công thức box tổng quát: (r // k) * k + c // k
    # TODO 3: dựng rows/cols/boxes mask + empties
    # TODO 4: backtrack() kiểu MRV giống Cách 4, nhưng dùng FULL_N
    pass

# Tự làm xong rồi mới mở test:
# mini = [list(row) for row in ["12.4", "..2.", "..13", "..4."]]
# exercise_solve_nxn(mini)
# assert ["".join(row) for row in mini] == ["1234", "4321", "2413", "3142"]
# b = [row[:] for row in EXAMPLE_BOARD]; exercise_solve_nxn(b); assert b == EXAMPLE_SOLUTION

## 20. Gợi ý theo mức

**Bài 1**
- Mức 1: tính riêng `r // 3` và `c // 3` trước.
- Mức 2: `box = (r // 3) * 3 + c // 3`; góc = `((r // 3) * 3, (c // 3) * 3)`.
- Mức 3: `(7, 2)` → `r // 3 = 2`, `c // 3 = 0` → box `6`, góc `(6, 0)`.

**Bài 2**
- Mức 1: box 4 là box giữa, gồm hàng 3–5, cột 3–5.
- Mức 2: ứng viên = `{1..9}` trừ đi hợp của 3 tập.
- Mức 3: hàng 4 có `{4, 8, 3, 1}`, cột 4 có `{7, 9, 6, 2, 1, 8}`, box 4 có `{6, 8, 3, 2}` → còn lại ... ?

**Bài 3**
- Mức 1: có 3 nơi có thể "đụng hàng": hàng, cột, box.
- Mức 2: một vòng `for i in range(9)` kiểm tra được cả hàng lẫn cột cùng lúc.
- Mức 3: box dùng 2 vòng `for i in range(start_r, start_r + 3)` / `for j in range(start_c, start_c + 3)`.

**Bài 4**
- Mức 1: mọi thứ thay đổi lúc "chọn" phải được hoàn tác lúc "bỏ chọn".
- Mức 2: kiểm tra `d in rows[r] or d in cols[c] or d in boxes[b]` thì `continue`.
- Mức 3: đừng quên `if backtrack(i + 1): return True` — không có dòng này lời giải bị gỡ mất (Lỗi 2).

**Bài 5**
- Mức 1: giống bước dựng set trong `build_sets`, chỉ thêm việc phát hiện trùng.
- Mức 2: kiểm tra "đã có chưa" **trước** khi `add`.
- Mức 3: `if v in rows[r] or v in cols[c] or v in boxes[b]: return False`.

**Bài 6**
- Mức 1: chỗ nào trong Cách 4 đang "cứng" số 3, số 9, `0x1FF`? Đó là chỗ cần tham số hóa.
- Mức 2: `FULL_N = (1 << n) - 1` có đúng `n` bit 1.
- Mức 3: copy `solve_sudoku_bitmask`, thay `box_id` bằng công thức theo `k`, thay `FULL` bằng `FULL_N`, bỏ biến đếm.

## 21. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1 và Bài 2

In [31]:
# Bài 1
for r, c in [(0, 8), (4, 4), (7, 2), (5, 6), (8, 8)]:
    print(f"({r},{c}) -> box {box_id(r, c)}, góc ({(r // 3) * 3},{(c // 3) * 3})")

# Bài 2
def answer_candidates(board, r, c):
    row = {v for v in board[r] if v != "."}
    col = {board[i][c] for i in range(9) if board[i][c] != "."}
    sr, sc = (r // 3) * 3, (c // 3) * 3
    box = {board[i][j] for i in range(sr, sr + 3) for j in range(sc, sc + 3) if board[i][j] != "."}
    cand = sorted(set(DIGITS) - row - col - box)
    print(f"ô ({r},{c}): hàng {sorted(row)}, cột {sorted(col)}, box {sorted(box)} -> ứng viên {cand}")
    return cand

assert answer_candidates(EXAMPLE_BOARD, 4, 4) == ["5"]
assert answer_candidates(EXAMPLE_BOARD, 6, 0) == ["1", "3", "9"]

(0,8) -> box 2, góc (0,6)
(4,4) -> box 4, góc (3,3)
(7,2) -> box 6, góc (6,0)
(5,6) -> box 5, góc (3,6)
(8,8) -> box 8, góc (6,6)
ô (4,4): hàng ['1', '3', '4', '8'], cột ['1', '2', '6', '7', '8', '9'], box ['2', '3', '6', '8'] -> ứng viên ['5']
ô (6,0): hàng ['2', '6', '8'], cột ['4', '5', '6', '7', '8'], box ['6'] -> ứng viên ['1', '3', '9']


Ô `(4, 4)` bị "ép" về `5` — đúng loại ô MRV sẽ chọn sớm. Ô `(6, 0)` có 3 ứng viên `1, 3, 9`; đáp án thật là `9`.

### Đáp án Bài 3, 4, 5, 6

Mỗi đáp án nằm trong một cell riêng, có assert chạy được.

In [32]:
# Đáp án Bài 3
def answer_is_valid_placement(board, r: int, c: int, d: str) -> bool:
    for i in range(9):
        if board[r][i] == d or board[i][c] == d:
            return False
    start_r, start_c = (r // 3) * 3, (c // 3) * 3
    for i in range(start_r, start_r + 3):
        for j in range(start_c, start_c + 3):
            if board[i][j] == d:
                return False
    return True

assert [d for d in DIGITS if answer_is_valid_placement(EXAMPLE_BOARD, 0, 2, d)] == ["1", "2", "4"]
assert answer_is_valid_placement(EXAMPLE_BOARD, 4, 4, "5") == True
assert answer_is_valid_placement(EXAMPLE_BOARD, 4, 4, "6") == False
print("Bài 3 PASS")

Bài 3 PASS


In [33]:
# Đáp án Bài 4
def answer_solve_sets(board) -> None:
    rows, cols, boxes, empties = build_sets(board)

    def backtrack(i: int) -> bool:
        if i == len(empties):
            return True
        r, c = empties[i]
        b = box_id(r, c)
        for d in DIGITS:
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            if backtrack(i + 1):
                return True
            board[r][c] = "."
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
        return False

    backtrack(0)

b = [row[:] for row in EXAMPLE_BOARD]
answer_solve_sets(b)
assert b == EXAMPLE_SOLUTION
b = [row[:] for row in HARD_BOARD]
answer_solve_sets(b)
assert is_valid_solution(b, HARD_BOARD)
print("Bài 4 PASS")

Bài 4 PASS


In [34]:
# Đáp án Bài 5
def answer_is_valid_sudoku(board) -> bool:
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // 3) * 3 + c // 3
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

assert answer_is_valid_sudoku(EXAMPLE_BOARD) == True
assert answer_is_valid_sudoku(EXAMPLE_SOLUTION) == True
bad = [row[:] for row in EXAMPLE_BOARD]; bad[0][2] = "6"        # trùng box 0
assert answer_is_valid_sudoku(bad) == False
bad = [row[:] for row in EXAMPLE_BOARD]; bad[8][0] = "5"        # trùng cột 0
assert answer_is_valid_sudoku(bad) == False
print("Bài 5 PASS")

Bài 5 PASS


In [35]:
# Đáp án Bài 6
def answer_solve_nxn(board) -> None:
    n = len(board)
    k = int(n ** 0.5)
    full_n = (1 << n) - 1
    rows, cols, boxes = [0] * n, [0] * n, [0] * n
    empties = []

    def bid(r, c):
        return (r // k) * k + c // k

    for r in range(n):
        for c in range(n):
            if board[r][c] == ".":
                empties.append((r, c))
            else:
                bit = 1 << (int(board[r][c]) - 1)
                rows[r] |= bit; cols[c] |= bit; boxes[bid(r, c)] |= bit

    def backtrack() -> bool:
        best, best_free, best_count = None, 0, n + 1
        for r, c in empties:
            if board[r][c] != ".":
                continue
            free = ~(rows[r] | cols[c] | boxes[bid(r, c)]) & full_n
            if free.bit_count() < best_count:
                best, best_free, best_count = (r, c), free, free.bit_count()
                if best_count <= 1:
                    break
        if best is None:
            return True
        r, c = best
        b = bid(r, c)
        free = best_free
        while free:
            bit = free & -free
            free ^= bit
            board[r][c] = str(bit.bit_length())
            rows[r] |= bit; cols[c] |= bit; boxes[b] |= bit
            if backtrack():
                return True
            board[r][c] = "."
            rows[r] ^= bit; cols[c] ^= bit; boxes[b] ^= bit
        return False

    backtrack()

mini = [list(row) for row in ["12.4", "..2.", "..13", "..4."]]
answer_solve_nxn(mini)
assert ["".join(row) for row in mini] == ["1234", "4321", "2413", "3142"]
b = [row[:] for row in EXAMPLE_BOARD]
answer_solve_nxn(b)
assert b == EXAMPLE_SOLUTION
b = [row[:] for row in EVIL_BOARD]
answer_solve_nxn(b)
assert is_valid_solution(b, EVIL_BOARD)
print("Bài 6 PASS:", ["".join(row) for row in mini])

Bài 6 PASS: ['1234', '4321', '2413', '3142']


## 22. Tổng kết

### Template backtracking dễ nhớ (Sudoku)

```python
def backtrack():
    cell = chọn ô trống (tốt nhất: ô ít ứng viên nhất)   # MRV
    if không còn ô trống:
        return True
    for d in ứng viên của cell:                          # kiểm tra nhanh: set / bitmask
        đặt d  (board + rows + cols + boxes)             # 1. chọn
        if backtrack():                                  # 2. đi tiếp
            return True
        gỡ d   (board + rows + cols + boxes)             # 3. bỏ chọn — đối xứng
    return False                                         # ngõ cụt -> cấp trên thử tiếp
```

**Ba câu cần nhớ:**

1. `box = (r // 3) * 3 + c // 3`.
2. **Chọn → đi tiếp → bỏ chọn**, và `if backtrack(): return True`.
3. Hai cách tăng tốc: **ít bước hơn** (MRV) và **mỗi bước rẻ hơn** (set → bitmask).

### Checklist trước khi submit

- [ ] Signature đúng: `def solveSudoku(self, board: List[List[str]]) -> None`, không `return` bảng.
- [ ] Sửa **in-place** (`board[r][c] = ...`), không gán lại `board = ...`.
- [ ] Ghi chữ số dạng **chuỗi** (`str(d)`), so sánh với `"5"` chứ không phải `5`.
- [ ] Công thức box đúng; nếu dùng set thì tạo bằng `[set() for _ in range(9)]`.
- [ ] Có `if backtrack(): return True` để giữ lời giải.
- [ ] Undo **đối xứng**: gỡ khỏi board và cả 3 cấu trúc phụ.
- [ ] Với bitmask: nhớ `& 0x1FF` sau `~`, và `bit.bit_length()` ra đúng chữ số `1..9`.
- [ ] Đã test ví dụ của đề + một bảng khó + bảng gần đầy.